In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!sudo apt-get install -qq libomp-dev
!pip install -qq faiss-gpu-cu12
!pip install -qq fastapi uvicorn pydantic pyngrok nest-asyncio gradio python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import numpy as np
import collections
import torch
import faiss
from unsloth import FastLanguageModel
from transformers import pipeline
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from transformers import AutoModelForQuestionAnswering
from transformers import TrainingArguments
from transformers import Trainer
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import TextStreamer

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [5]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        return_tensors='pt'
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)

    return cls_pooling(model_output)

In [6]:

embeddings_dataset = load_dataset('MinhQuy24/SQuAD_QA_Vector_Database', split="train")
index_file_path = hf_hub_download(
    repo_id='MinhQuy24/SQuAD_QA_Vector_Database',
    filename="my_index.faiss",
    repo_type="dataset"
)
embeddings_dataset.load_faiss_index('question_embedding', index_file_path)

README.md:   0%|          | 0.00/508 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/285M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/86821 [00:00<?, ? examples/s]

my_index.faiss:   0%|          | 0.00/267M [00:00<?, ?B/s]

In [7]:
MODEL_NAME = 'MinhQuy24/llama3.2_3B_SQuAD_QA'
max_seq_length = 2048
dtype = None
load_in_4bit = True

model_QA, tokenizer_QA = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
text_streamer = TextStreamer(tokenizer_QA, skip_prompt = True)
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/48.7M [00:00<?, ?B/s]

Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [9]:
def format_test_squad(context,question):
    return {"text": f"""<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
You are a helpful AI assistant providing accurate answers based on the given context.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Context: {context}
Question: {question}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Answer: """}

In [10]:
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok

In [11]:
class ChatRequest(BaseModel):
    question: str
    top_k: int = 3

class QAItem(BaseModel):
    answer: str
    context: str

class ChatResponse(BaseModel):
    question: str
    results: list[QAItem]

app = FastAPI(title="SQuAD RAG API")

In [12]:
@app.post("/ask", response_model=ChatResponse)
async def ask_bot(request: ChatRequest):
    try:
        input_quest_embedding = get_embeddings([request.question]).cpu().detach().numpy()
        scores, samples = embeddings_dataset.get_nearest_examples(
            'question_embedding', input_quest_embedding, k=request.top_k
        )

        contexts = samples['context']
        results = []

        for context in contexts:
            formatted_prompt = format_test_squad(context, request.question)["text"]
            inputs = tokenizer_QA(formatted_prompt, return_tensors="pt").to("cuda")

            outputs = model_QA.generate(
                input_ids=inputs["input_ids"],
                max_new_tokens=64,
                use_cache=True,
                temperature=0.3,
                min_p=0.1,
                pad_token_id=tokenizer_QA.eos_token_id
            )

            full_text = tokenizer_QA.decode(outputs[0], skip_special_tokens=False)
            answer = full_text.split("Answer: ")[-1].replace('<|eot_id|>', '').strip()

            results.append(QAItem(answer=answer, context=context))

        return ChatResponse(
            question=request.question,
            results=results
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [13]:
import asyncio
nest_asyncio.apply()

ngrok.set_auth_token('3AdwuLmFnN8RVo0gdqcn8VV4NxR_2TNjfnh8awDXs31NjfRJW')

public_url = ngrok.connect(8000)
print("="*50)
print(f"🚀 API CỦA BẠN ĐÃ ONLINE TẠI: {public_url.public_url}")
print(f"👉 Bấm vào link này để test giao diện API: {public_url.public_url}/docs")
print("="*50)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

asyncio.create_task(server.serve())

🚀 API CỦA BẠN ĐÃ ONLINE TẠI: https://kathie-pseudoascetical-geoffrey.ngrok-free.dev
👉 Bấm vào link này để test giao diện API: https://kathie-pseudoascetical-geoffrey.ngrok-free.dev/docs


<Task pending name='Task-1' coro=<Server.serve() running at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:77>>

In [16]:
import gradio as gr
import requests

def ask_rag_system(api_url, question, top_k):
    if not api_url or not question:
        return "⚠️ Please enter both the API URL and your question!"

    api_url = api_url.rstrip("/")
    endpoint = f"{api_url}/ask"

    payload = {
        "question": question,
        "top_k": int(top_k)
    }

    try:
        response = requests.post(endpoint, json=payload, timeout=60)
        response.raise_for_status()

        data = response.json()

        output_markdown = f"### 🎯 Question: {data['question']}\n\n"

        for idx, item in enumerate(data['results']):
            output_markdown += f"#### 🔍 Result {idx + 1}\n"
            output_markdown += f"**🤖 Llama 3 Answer:** {item['answer']}\n\n"
            output_markdown += f"> **📄 Extracted Context (FAISS):** *{item['context']}*\n\n"
            output_markdown += "---\n"

        return output_markdown

    except requests.exceptions.RequestException as e:
        return f"❌ **Connection Error:** `{e}`\n\n*Make sure the backend is running and you have pasted the correct Ngrok link!*"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🧠 AI Question Answering (RAG System)")
    gr.Markdown("This system combines **FAISS** (retrieval) and **Llama 3** (generation).")

    with gr.Row():
        with gr.Column(scale=1):
            api_url_input = gr.Textbox(
                label="🌐 API URL (Ngrok URL)",
                placeholder="e.g., https://abcd-123.ngrok-free.app",
                info="Paste the Ngrok link from your Backend here."
            )

            question_input = gr.Textbox(
                label="❓ Enter your question",
                placeholder="e.g., When did Beyonce start becoming popular?",
                lines=3
            )

            top_k_input = gr.Slider(
                minimum=1, maximum=5, value=3, step=1,
                label="Number of results (Top K)",
                info="Number of context snippets FAISS will retrieve."
            )

            submit_btn = gr.Button("🚀 Submit Question", variant="primary")

        with gr.Column(scale=2):
            output_display = gr.Markdown(
                label="Response Output",
                value="*Results will appear here...*"
            )
    submit_btn.click(
        fn=ask_rag_system,
        inputs=[api_url_input, question_input, top_k_input],
        outputs=output_display
    )

/tmp/ipykernel_2619/2096751883.py:44: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


In [17]:
if __name__ == "__main__":
    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ef093ed88a0adb57c4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
